# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### 1.1 Playbook Architecture: From Model Probability to Actionable Queue

A machine learning ranking probability is a decision-support signal, not an editorial command. To be useful to a human content team, every flagged item must map to a specific **content archetype**, a concrete **recommended action**, and a transparent **reason code**.

#### Archetype-to-Action Mapping Matrix

| Archetype | Defining Criteria (Snapshot Metrics) | Primary Action Label | Reason Code | Estimated Editorial Effort |
|---|---|---|---|---|
| **Striking-Distance Opportunity** | Position 4–20, high impressions ($\ge$ 3,000), CTR $\le$ 0.5% | `review_title_meta_and_intent` | `striking_distance_low_ctr` | Low (~15 mins): Rewrite title tag, meta description, and match SERP intent |
| **Decaying High-Demand Core** | High impressions ($\ge$ 5,000), model decline prob $\ge$ 0.60 | `comprehensive_content_refresh` | `high_demand_active_decay` | High (~1–2 hours): Update outdated facts, expand thin sections, add media |
| **High Visibility CTR Deficit** | Page 1 position ($\le$ 3), high impressions, CTR below expected benchmark | `serp_snippet_optimization` | `top3_underperforming_ctr` | Low (~15 mins): Test SERP headline & structured data snippets |
| **Thin / Zombie Content** | Low impressions (< 100), high age (> 180d), low engagement | `evaluate_consolidation_or_pruning` | `low_demand_zero_engagement` | Moderate (~30 mins): Evaluate 301 redirect, consolidation, or canonicalization |
| **Stable Performer** | High impressions, low decline probability (< 0.40), healthy CTR | `monitor_only` | `stable_healthy_baseline` | None (0 mins): Do not touch |

#### The Decay & Refresh Insight
Our signal audit in Week 4 demonstrated that staleness alone does not predict traffic decline: pages aged 181+ days exhibited a lower observed decline rate (47.1%) than pages aged 91–180 days (61.1%). Refreshing pages solely because of calendar age is an operational antipattern. Content refreshes must be prioritized where **observed velocity drops intersect with demonstrable search demand**.

#### Cost / Value Triage Framework
Editorial capacity is finite. Assuming an editor can thoroughly review and update 3–4 pages per day:
- **High Expected Value:** Flagging a page with 50,000 impressions at position 5.4 where a 0.2% CTR lift delivers 100 incremental clicks/month.
- **Negative Expected Value:** Flagging a page with 40 impressions where a full refresh costs $150 of editorial time for a negligible traffic upside.
- The queue is therefore prioritized by **potential impact** (impressions and ranking proximity) conditioned on **model decline risk**.

In [1]:
import pandas as pd
from pathlib import Path

csv_path = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(csv_path)

print(f"Loaded {len(df):,} content items across {df['client_id'].nunique()} clients.")
df.head()

Loaded 30,000 content items across 32 clients.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Define target outcome
df["trend_outcome"] = (df["trend_direction"] == "down").astype(int)

# 2. Exclude identifiers, label derivatives, and short-window leakages (from w03 & w06)
prohibited_columns = {
    "trend_outcome", "trend_direction", "trend_pct",
    "client_id", "content_id", "provider_used", "model_used",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
}
feature_columns = [col for col in df.columns if col not in prohibited_columns]

# 3. Fit pipeline on client-grouped split (matching w05 & w06)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(
    splitter.split(df, y=df["trend_outcome"], groups=df["client_id"])
)

train_df = df.iloc[train_idx]
x_train = train_df[feature_columns]
y_train = train_df["trend_outcome"]

numeric_cols = x_train.select_dtypes(include=["number", "bool"]).columns
categorical_cols = x_train.select_dtypes(include=["object"]).columns

preprocessing = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]),
        numeric_cols,
    ),
    (
        "categorical",
        Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]),
        categorical_cols,
    ),
])

model = Pipeline([
    ("preprocess", preprocessing),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(x_train, y_train)

# 4. Generate model probabilities across the portfolio
df["model_probability"] = model.predict_proba(df[feature_columns])[:, 1]
df["is_test_client"] = df.index.isin(test_idx)

# 5. Archetype & Action Mapping (based on Section 1.1 matrix)
cond_decaying = (df["impressions_90d"] >= 5000) & (df["model_probability"] >= 0.60)
cond_striking = (
    (df["avg_position"].between(4, 20))
    & (df["impressions_90d"] >= 3000)
    & (df["ctr"] <= 0.5)
)
cond_top3_ctr = (
    (df["avg_position"].between(1, 3))
    & (df["impressions_90d"] >= 3000)
    & (df["ctr"] <= 0.8)
)
cond_zombie = (df["impressions_90d"] < 100) & (df["content_age_days"] > 180)

archetype_conditions = [cond_decaying, cond_striking, cond_top3_ctr, cond_zombie]

archetype_choices = [
    "Decaying High-Demand Core",
    "Striking-Distance Opportunity",
    "High Visibility CTR Deficit",
    "Thin / Zombie Content",
]

action_choices = [
    "comprehensive_content_refresh",
    "review_title_meta_and_intent",
    "serp_snippet_optimization",
    "evaluate_consolidation_or_pruning",
]

reason_choices = [
    "high_demand_active_decay",
    "striking_distance_low_ctr",
    "top3_underperforming_ctr",
    "low_demand_zero_engagement",
]

df["archetype"] = np.select(
    archetype_conditions, archetype_choices, default="Stable Performer"
)
df["action_label"] = np.select(
    archetype_conditions, action_choices, default="monitor_only"
)
df["reason_code"] = np.select(
    archetype_conditions, reason_choices, default="stable_healthy_baseline"
)

# 6. Priority score calculation: Expected Value = Impressions * Model Decline Probability
df["priority_score"] = (
    (df["impressions_90d"] * df["model_probability"])
    .where(df["action_label"] != "monitor_only", 0)
    .round(2)
)

# 7. Build ranked actionable queue
queue_cols = [
    "content_id",
    "priority_score",
    "archetype",
    "action_label",
    "reason_code",
    "model_probability",
    "impressions_90d",
    "avg_position",
    "ctr",
    "is_test_client",
]

ranked_queue = (
    df[df["action_label"] != "monitor_only"][queue_cols]
    .sort_values("priority_score", ascending=False)
    .reset_index(drop=True)
)

print(f"Total inventory: {len(df):,} items")
print(
    f"Actionable candidates identified: {len(ranked_queue):,} "
    f"({len(ranked_queue)/len(df)*100:.1f}%)"
)
print("\nInventory Breakdown by Archetype:")
print(df["archetype"].value_counts())

print("\n--- Top 10 Priority Triage Queue ---")
ranked_queue.head(10)

Total inventory: 30,000 items
Actionable candidates identified: 10,638 (35.5%)

Inventory Breakdown by Archetype:
archetype
Stable Performer                 19362
Thin / Zombie Content             4743
Decaying High-Demand Core         2950
Striking-Distance Opportunity     2744
High Visibility CTR Deficit        201
Name: count, dtype: int64

--- Top 10 Priority Triage Queue ---


,content_id,priority_score,archetype,action_label,reason_code,model_probability,impressions_90d,avg_position,ctr,is_test_client
0,content_5fe46e04994d,348369.15,Decaying High-Demand Core,comprehensive_content_refresh,high_demand_active_decay,0.672898,517715,4.2,0.14,True
1,content_2cb567c3c89b,310048.27,Decaying High-Demand Core,comprehensive_content_refresh,high_demand_active_decay,0.622928,497727,22.2,0.10,False
2,content_2dba2b1f9536,308663.74,Decaying High-Demand Core,comprehensive_content_refresh,high_demand_active_decay,0.696076,443434,27.9,0.21,False
3,content_8e7ba84a972b,288192.56,Decaying High-Demand Core,comprehensive_content_refresh,high_demand_active_decay,0.999191,288426,4.8,0.92,False
4,content_36ff89c8214e,196817.80,Decaying High-Demand Core,comprehensive_content_refresh,high_demand_active_decay,0.666960,295097,7.3,0.05,False
5,content_cb112fce36be,195256.31,Decaying High-Demand Core,comprehensive_content_refresh,high_demand_active_decay,0.630042,309910,5.6,0.16,False
6,content_c84a0ab98e90,191975.48,Decaying High-Demand Core,comprehensive_content_refresh,high_demand_active_decay,0.859832,223271,7.8,0.03,True
7,content_aaef01a50def,188838.45,Striking-Distance Opportunity,review_title_meta_and_intent,striking_distance_low_ctr,0.365181,517109,5.4,0.25,False
8,content_66b4046cc144,185379.95,Decaying High-Demand Core,comprehensive_content_refresh,high_demand_active_decay,0.852655,217415,26.6,0.03,False
9,content_b28d1efd668f,182183.66,Decaying High-Demand Core,comprehensive_content_refresh,high_demand_active_decay,0.635654,286608,26.2,0.06,False


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### 2.1 Intended Use and Operating Context

This action playbook is a **decision-support triage tool** built for editorial teams, content marketing leads, and SEO managers. Its explicit function is to eliminate editorial guesswork by organizing large content inventories into an evidence-backed review backlog.

- **Primary Users:** Content editors, SEO managers, and digital strategists managing inventories of hundreds to thousands of articles.
- **Intended Decision:** Determining which pages to inspect first during content refresh sprints, and scoping the level of editorial intervention required (e.g., lightweight title/snippet rewriting versus comprehensive article overhaul).
- **Core Value Proposition:** Shielding editorial bandwidth by automatically filtering out ~64.5% of stable/low-priority inventory and directing human effort toward items with the largest potential traffic at risk.

### 2.2 Known Limits and Operational Boundaries

1. **Observational, Non-Causal Nature:** The model and rules identify statistical associations present in historical snapshot data. They do **not** prove that refreshing a page will guarantee rank improvement or traffic recovery.
2. **External SERP Layout Confounders:** The dataset contains on-page and aggregate search metrics, but lacks direct SERP layout data (such as Google AI Overviews, sponsored ad clusters, video carousels, or local map packs). A top-3 page with a low CTR may be losing clicks to search features that on-page metadata cannot fix.
3. **Historical Window Overlap:** As established in our Week 6 validation audit, the 90-day aggregate performance metrics overlap with the 30-day window used to define `trend_outcome`. Consequently, predictions represent retrospective classification of observed performance rather than clairvoyant forecasting.
4. **Low-Data Volatility Threshold:** Metrics from pages with fewer than 500 impressions or volatile rankings exhibit extreme variance. Decisions on low-volume pages carry wider uncertainty intervals.
5. **Cross-Client Portfolio Heterogeneity:** Model weights reflect average patterns across the 32 portfolio clients. Niche domains with idiosyncratic publishing rhythms or specialized technical formats may require custom threshold tuning.

In [3]:
# Quantifying operational limits: Confidence Tiers across the actionable queue
# High confidence requires substantial impression volume and established position data.

queue_limits = ranked_queue.copy()

confidence_conditions = [
    (queue_limits["impressions_90d"] >= 10000) & (queue_limits["avg_position"] > 0),
    (queue_limits["impressions_90d"].between(1000, 9999)) & (queue_limits["avg_position"] > 0),
    (queue_limits["impressions_90d"] < 1000) | (queue_limits["avg_position"] == 0),
]
confidence_labels = ["High Confidence", "Medium Confidence", "Low Confidence (High Variance)"]

queue_limits["confidence_tier"] = np.select(
    confidence_conditions, confidence_labels, default="Medium Confidence"
)

limits_summary = (
    queue_limits.groupby(["archetype", "confidence_tier"])
    .size()
    .unstack(fill_value=0)
)

print("--- Actionable Queue Segmented by Archetype and Operational Confidence Tier ---")
print(limits_summary)

low_data_share = (queue_limits["confidence_tier"] == "Low Confidence (High Variance)").mean() * 100
print(f"\nShare of actionable recommendations in low-confidence tier: {low_data_share:.1f}%")
print("Guidance: Human reviewers should prioritize High and Medium confidence items first.")

--- Actionable Queue Segmented by Archetype and Operational Confidence Tier ---
confidence_tier                High Confidence  \
archetype                                        
Decaying High-Demand Core                 1319   
High Visibility CTR Deficit                 97   
Striking-Distance Opportunity             1139   
Thin / Zombie Content                        0   

confidence_tier                Low Confidence (High Variance)  \
archetype                                                       
Decaying High-Demand Core                                   0   
High Visibility CTR Deficit                                 0   
Striking-Distance Opportunity                               0   
Thin / Zombie Content                                    4743   

confidence_tier                Medium Confidence  
archetype                                         
Decaying High-Demand Core                   1631  
High Visibility CTR Deficit                  104  
Striking-Distance Opport

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### 3.1 The Human Review Protocol (Pre-Action Checklist)

Every flagged recommendation must pass a four-step editorial review before any changes are pushed to production:

1. **Live SERP & Intent Verification:** Search the primary keyword in an incognito window. Verify whether search intent has shifted (e.g., from informational comparison to commercial product listing) and assess whether Google AI Overviews or rich cards dominate above-the-fold space.
2. **Factual and Technical Integrity Check:** Audit the article for outdated facts, expired product references, dead outbound links, or changed company policies.
3. **Snippet Truncation & CTR Diagnosis:** Inspect the current title tag length (< 60 characters) and meta description (< 155 characters) to ensure the search snippet communicates a compelling, differentiated value proposition.
4. **Recency Verification:** Confirm the page has not undergone a refresh in the last 30–60 days to allow search engines sufficient time to index and stabilize previous updates.

### 3.2 The Non-Negotiable "NO-GO" List (What Must NEVER Be Automated)

- **NEVER Auto-Deploy Generative AI Content Rewrites:** Never allow autonomous LLM pipelines to rewrite and publish articles directly to live sites. Unreviewed AI text risks hallucinations, tone mismatch, and Google helpful-content penalties.
- **NEVER Programmatically Bulk-Prune URLs:** Never automatically 404, delete, or de-index URLs flagged under `Thin / Zombie Content`. Automated pruning risks destroying valuable historical backlinks, breaking internal site navigation, or deleting regulatory documentation.
- **NEVER Overwrite High-Converting Commercial Metadata:** Pages with transactional intent or active conversion value must not be touched by bulk templated updates without senior commercial signoff.
- **NEVER Treat Seasonal Fluctuations as Content Failure:** Predictable calendar drops (such as holiday retail in January or tax content in May) must not trigger emergency rewrites.

In [4]:
# Implementing automated safety guards and review flags in the triage queue
df["requires_human_intent_audit"] = df["action_label"] != "monitor_only"

# Guard 1: High-Risk Prune Guard - Thin content with residual sessions/clicks must not be deleted
df["prune_safety_guard"] = (
    (df["archetype"] == "Thin / Zombie Content")
    & ((df["sessions_90d"] > 0) | (df["clicks_90d"] > 0))
)

# Guard 2: High-Value Brand Guard - High-impression or transactional pages require Senior Editor signoff
df["senior_signoff_required"] = (
    (df["impressions_90d"] >= 50000) | (df["main_intent"] == "transactional")
) & (df["action_label"] != "monitor_only")

# Guard 3: Recency Guard - Content updated within 60 days must be cooled down
df["recent_update_cooldown"] = (df["days_since_last_update"] <= 60) & (
    df["action_label"] != "monitor_only"
)

safety_audit_summary = pd.DataFrame({
    "Safety Guard / Operational Policy": [
        "Pre-Action Human Intent Audit (All Actionable)",
        "Prune Safety Guard (Has Residual Sessions/Clicks)",
        "Senior Editor Signoff Required (High Value / Commercial)",
        "Recent Update Cooldown (Updated <= 60d ago)",
    ],
    "Flagged Items": [
        df["requires_human_intent_audit"].sum(),
        df["prune_safety_guard"].sum(),
        df["senior_signoff_required"].sum(),
        df["recent_update_cooldown"].sum(),
    ],
    "Required Action Before Deployment": [
        "Complete 4-point human checklist",
        "Consolidate/redirect only; DO NOT delete or 404",
        "Mandatory Senior Content Lead review before release",
        "Defer review; allow search engines time to index prior change",
    ],
})

print("--- Editorial Safety Guardrails Across Portfolio ---")
safety_audit_summary

--- Editorial Safety Guardrails Across Portfolio ---


,Safety Guard / Operational Policy,Flagged Items,Required Action Before Deployment
0,Pre-Action Human Intent Audit (All Actionable),10638,Complete 4-point human checklist
1,Prune Safety Guard (Has Residual Sessions/Clicks),4743,Consolidate/redirect only; DO NOT delete or 404
2,Senior Editor Signoff Required (High Value / C...,2252,Mandatory Senior Content Lead review before re...
3,Recent Update Cooldown (Updated <= 60d ago),7375,Defer review; allow search engines time to ind...


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### 4.1 Monitoring and Staleness Indicators

Content recommendation systems face inevitable drift due to search engine algorithm updates, competitor actions, and evolving query demand. To ensure recommendations remain trustworthy, the playbook incorporates continuous monitoring:

- **Queue Expiration (Time Decay):** Actionable recommendations that remain unreviewed for more than 60 days expire automatically and must be re-scored against fresh search performance data.
- **Core Update Volatility:** Major confirmed Google core updates trigger an immediate freeze and re-audit of the striking-distance and CTR deficit queues.
- **Editorial Divergence Feedback:** Reviewers log feedback on each inspected recommendation. If the editorial rejection rate exceeds 25%, rule thresholds and model calibration must be reviewed.

### 4.2 Concrete Retrain and Recalibration Triggers

| Trigger Category | Metric & Threshold | Operational Response |
|---|---|---|
| **Data Drift (Distributional Shift)** | Population Stability Index (PSI) > 0.25 on `impressions_90d` or `avg_position` | Trigger feature audit; recalibrate scalers and archetype cutoffs |
| **Precision Degradation** | Held-out Precision@20 drops below 0.50 (falling to random baseline rate) | Retrain logistic regression model on fresh 90-day warehouse partition |
| **Editorial Rejection Spike** | Editorial rejection rate exceeds 25% over 50 consecutive audited items | Review false-positive reason codes; tighten impression and position thresholds |
| **Scheduled Maintenance** | Calendar interval reaches 90 days | Scheduled quarterly re-fit using rolling multi-client data contract |

In [5]:
# Production monitoring drift baseline: reference quantiles for key signals
monitoring_features = ["impressions_90d", "avg_position", "ctr", "model_probability"]

baseline_drift_reference = df[monitoring_features].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
).T[["mean", "std", "5%", "50%", "95%"]].round(3)

baseline_drift_reference.columns = [
    "baseline_mean", "baseline_std", "p05_ref", "median_ref", "p95_ref"
]

monitoring_spec = pd.DataFrame([
    {
        "Monitoring Metric": "Precision@20 (60-day lag)",
        "Inspection Cadence": "Monthly cohort review",
        "Alert Threshold": "< 0.50 precision",
        "Operational Response": "Retrain logistic regression model on updated grouped split",
    },
    {
        "Monitoring Metric": "Editorial Rejection Rate",
        "Inspection Cadence": "Weekly sprint retrospective",
        "Alert Threshold": "> 25% rejection rate",
        "Operational Response": "Recalibrate archetype thresholds & reason code rules",
    },
    {
        "Monitoring Metric": "Feature Drift (PSI)",
        "Inspection Cadence": "Monthly warehouse refresh",
        "Alert Threshold": "PSI >= 0.25 on key metrics",
        "Operational Response": "Re-fit preprocessing transformers on new data partition",
    },
    {
        "Monitoring Metric": "Unreviewed Recommendation Age",
        "Inspection Cadence": "Continuous",
        "Alert Threshold": "> 60 days in queue",
        "Operational Response": "Expire recommendation; re-score under fresh 90d window",
    },
])

print("--- Production Monitoring Drift Baseline Reference ---")
print(baseline_drift_reference)

print("\n--- Operational Retrain & Recalibration Trigger Matrix ---")
monitoring_spec

--- Production Monitoring Drift Baseline Reference ---
                   baseline_mean  baseline_std  p05_ref  median_ref    p95_ref
impressions_90d         5200.366     16838.020    2.000     731.000  22996.500
avg_position              16.342        15.217    1.700      10.800     48.200
ctr                        0.511         3.279    0.000       0.070      1.090
model_probability          0.551         0.194    0.176       0.581      0.819

--- Operational Retrain & Recalibration Trigger Matrix ---


,Monitoring Metric,Inspection Cadence,Alert Threshold,Operational Response
0,Precision@20 (60-day lag),Monthly cohort review,< 0.50 precision,Retrain logistic regression model on updated g...
1,Editorial Rejection Rate,Weekly sprint retrospective,> 25% rejection rate,Recalibrate archetype thresholds & reason code...
2,Feature Drift (PSI),Monthly warehouse refresh,PSI >= 0.25 on key metrics,Re-fit preprocessing transformers on new data ...
3,Unreviewed Recommendation Age,Continuous,> 60 days in queue,Expire recommendation; re-score under fresh 90...


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### 5.1 Exported Research Artifacts

To support the recommendations and analysis sections of our upcoming research paper, this section exports all core outputs:

1. **`work/outputs/action_playbook_queue.csv`:** The complete ranked actionable queue containing priority scores, archetypes, action labels, reason codes, safety guards, and test-client indicators. (Excluded from git tracking by repository data leak rules).
2. **`work/outputs/action_playbook_summary.json`:** Structured summary metrics capturing inventory sizes, actionable proportions, archetype breakdowns, and safety guard tallies.
3. **`work/figures/playbook_archetype_distribution.png`:** High-resolution horizontal bar chart illustrating portfolio segmentation across operational archetypes.
4. **`work/figures/playbook_priority_queue_top20.png`:** Publication-quality bar chart visualizing the top 20 actionable opportunities by priority score and archetype.

In [6]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

# 1. Ensure export directories exist
outputs_dir = Path("../outputs")
figures_dir = Path("../figures")
outputs_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# 2. Export complete actionable queue to CSV
export_columns = [
    "content_id", "priority_score", "archetype", "action_label", "reason_code",
    "model_probability", "impressions_90d", "avg_position", "ctr",
    "prune_safety_guard", "senior_signoff_required", "recent_update_cooldown", "is_test_client"
]
queue_export_df = df[df["action_label"] != "monitor_only"][export_columns].sort_values(
    "priority_score", ascending=False
).reset_index(drop=True)

queue_csv_path = outputs_dir / "action_playbook_queue.csv"
queue_export_df.to_csv(queue_csv_path, index=False)

# 3. Export summary metrics to JSON
summary_metrics = {
    "total_inventory_items": int(len(df)),
    "actionable_candidates_count": int(len(queue_export_df)),
    "actionable_candidate_pct": round(float(len(queue_export_df) / len(df) * 100), 2),
    "archetype_breakdown": {str(k): int(v) for k, v in df["archetype"].value_counts().items()},
    "action_label_breakdown": {str(k): int(v) for k, v in df["action_label"].value_counts().items()},
    "safety_guards": {
        "prune_safety_guard_count": int(df["prune_safety_guard"].sum()),
        "senior_signoff_required_count": int(df["senior_signoff_required"].sum()),
        "recent_update_cooldown_count": int(df["recent_update_cooldown"].sum()),
    },
    "model_held_out_precision_at_20": 0.600,
    "baseline_held_out_precision_at_20": 0.400,
}

metrics_json_path = outputs_dir / "action_playbook_summary.json"
with open(metrics_json_path, "w") as f:
    json.dump(summary_metrics, f, indent=2)

# 4. Generate Figure 1: Portfolio Archetype Distribution
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig1, ax1 = plt.subplots(figsize=(9, 4.8))
archetype_order = [
    "Stable Performer", "Thin / Zombie Content", "Decaying High-Demand Core",
    "Striking-Distance Opportunity", "High Visibility CTR Deficit"
]
archetype_counts = df["archetype"].value_counts().reindex(archetype_order)
colors = ["#2b5c8f", "#d95f02", "#7570b3", "#1b9e77", "#e7298a"]
bars1 = ax1.barh(archetype_counts.index, archetype_counts.values, color=colors, edgecolor="black", alpha=0.85)
ax1.set_xlabel("Number of Content Items", fontsize=11, fontweight="bold")
ax1.set_title("Portfolio Content Triage Breakdown by Operational Archetype (n=30,000)", fontsize=12, fontweight="bold", pad=12)

for bar in bars1:
    width = bar.get_width()
    pct = width / len(df) * 100
    ax1.text(width + 250, bar.get_y() + bar.get_height() / 2, f"{width:,} ({pct:.1f}%)", va="center", fontsize=9.5)

ax1.set_xlim(0, max(archetype_counts.values) * 1.18)
fig1_path = figures_dir / "playbook_archetype_distribution.png"
plt.tight_layout()
plt.savefig(fig1_path, dpi=300)
plt.close(fig1)

# 5. Generate Figure 2: Top-20 Priority Queue Distribution
top20_df = queue_export_df.head(20).copy()
fig2, ax2 = plt.subplots(figsize=(10, 5.5))
archetype_color_map = {
    "Decaying High-Demand Core": "#7570b3",
    "Striking-Distance Opportunity": "#1b9e77",
    "High Visibility CTR Deficit": "#e7298a",
    "Thin / Zombie Content": "#d95f02",
}
bar_colors = [archetype_color_map.get(a, "#333333") for a in top20_df["archetype"]]
indices = np.arange(len(top20_df))
bars2 = ax2.bar(indices, top20_df["priority_score"] / 1000, color=bar_colors, edgecolor="black", alpha=0.85)
ax2.set_xticks(indices)
ax2.set_xticklabels(top20_df["content_id"].str.slice(0, 15), rotation=45, ha="right", fontsize=9)
ax2.set_ylabel("Priority Triage Score (in thousands)", fontsize=11, fontweight="bold")
ax2.set_title("Top-20 Content Action Triage Queue (Prioritized by Expected Impact)", fontsize=12, fontweight="bold", pad=12)

# Legend
unique_top_archetypes = [a for a in archetype_color_map if a in top20_df["archetype"].unique()]
handles = [plt.Rectangle((0, 0), 1, 1, color=archetype_color_map[k], ec="k") for k in unique_top_archetypes]
ax2.legend(handles, unique_top_archetypes, loc="upper right", frameon=True)

fig2_path = figures_dir / "playbook_priority_queue_top20.png"
plt.tight_layout()
plt.savefig(fig2_path, dpi=300)
plt.close(fig2)

print("--- Exported Artifacts for Research Paper ---")
print(f"1. Queue CSV:    {queue_csv_path.resolve()} ({queue_csv_path.stat().st_size:,} bytes, {len(queue_export_df):,} rows)")
print(f"2. Metrics JSON: {metrics_json_path.resolve()} ({metrics_json_path.stat().st_size:,} bytes)")
print(f"3. Figure 1:     {fig1_path.resolve()} ({fig1_path.stat().st_size:,} bytes)")
print(f"4. Figure 2:     {fig2_path.resolve()} ({fig2_path.stat().st_size:,} bytes)")

--- Exported Artifacts for Research Paper ---
1. Queue CSV:    D:\Astronial\College\FlyRank internship\Code\My-Flyrank\work\outputs\action_playbook_queue.csv (1,770,933 bytes, 10,638 rows)
2. Metrics JSON: D:\Astronial\College\FlyRank internship\Code\My-Flyrank\work\outputs\action_playbook_summary.json (821 bytes)
3. Figure 1:     D:\Astronial\College\FlyRank internship\Code\My-Flyrank\work\figures\playbook_archetype_distribution.png (132,510 bytes)
4. Figure 2:     D:\Astronial\College\FlyRank internship\Code\My-Flyrank\work\figures\playbook_priority_queue_top20.png (245,863 bytes)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.